# 🍳 The CDS Cookbook, in a notebook

A runnable companion to [`docs/cookbook.md`](https://furox-art.github.io/scientific-computing-system/cookbook/) — the same problem-oriented recipes, grouped by domain and executable as-is.

Everything below runs on the **zero-dependency core**: pure Python, no NumPy, no SciPy, no extra installs. Switch the kernel to any Python 3.11+ environment with `scientific-computing-system` installed and press **Run All**.

In [ ]:
import cds

print("Scientific Computing System v" + cds.__version__)

## 1. Core data models

`Domain`, `Hypothesis` and `HypothesisStatus` are the research artifacts that travel through every higher-level module. A hypothesis needs an `id`, a `statement`, a `domain` and a `research_question`; testable consequences go in `predictions`.

In [ ]:
from cds.core.models import Domain, Hypothesis, HypothesisStatus

h = Hypothesis(
    id="H-001",
    statement="Increasing reactor temperature from 410K to 415K raises yield by >5%.",
    domain=Domain.CHEMISTRY,
    research_question="How does temperature affect yield?",
    assumptions=["reaction rate is Arrhenius in this range", "no side reactions"],
    predictions=["yield_delta > 0.05"],
    confidence=0.7,
    status=HypothesisStatus.TESTABLE,
)
print(h.predictions)  # the testable consequences
print(h.to_markdown())  # a structured write-up

## 2. Math utilities

Linear algebra and calculus primitives, written out in readable Python.

In [ ]:
from cds.math_utils import solve_linear

# 2x +  y = 5
#  x + 3y = 10
A = [[2.0, 1.0], [1.0, 3.0]]
b = [5.0, 10.0]
print(solve_linear(A, b))  # [1.0, 3.0]

In [ ]:
from cds.math_utils import derivative, gram_schmidt, integral, power_iteration

M = [[4.0, 1.0], [1.0, 3.0]]
eigval, eigvec = power_iteration(M, max_iter=200)
print("dominant eigenvalue:", round(eigval, 6))  # ~4.618

print("d/dx x^3 at 2 :", round(derivative(lambda x: x**3, 2.0), 6))  # ~12.0
print("int x^2 0..3   :", round(integral(lambda x: x**2, 0.0, 3.0), 6))  # ~9.0

basis = [[1.0, 1.0, 0.0], [0.0, 1.0, 1.0]]
print("orthonormal basis:", [[round(v, 6) for v in vec] for vec in gram_schmidt(basis)])

## 3. Probability

Densities, PMFs and a sampler for quick simulations.

In [ ]:
from cds.probability import (
    binomial_pmf,
    exponential_pdf,
    gaussian_pdf,
    poisson_pmf,
    uniform_sample,
)

print(round(gaussian_pdf(1.0, mu=0.0, sigma=1.0), 4))  # ~0.2420
print(round(exponential_pdf(0.5, lam=2.0), 4))  # ~0.7358
print(round(binomial_pmf(3, n=10, p=0.5), 4))  # ~0.1172
print(round(poisson_pmf(6, lam=4.0), 4))  # ~0.1042
print([round(v, 3) for v in uniform_sample(0.0, 10.0, n=5)])

## 4. Scientific formulas and constants

Physical constants plus the formulas that chain them together.

In [ ]:
from cds.scientific import de_broglie_wavelength, get_constant, photon_energy

h = get_constant("h")  # Planck constant
c = get_constant("c")
E = photon_energy(h * c / 500e-9)  # a 500 nm photon
print("E =", f"{E:.3e}", "J")
print("lambda_dB(electron, 1e6 m/s) =", f"{de_broglie_wavelength(9.109e-31, 1e6):.3e}", "m")

In [ ]:
from cds.scientific import escape_velocity, gravitational_force, schwarzschild_radius

print("Earth-Moon pull :", f"{gravitational_force(5.972e24, 7.348e22, 3.844e8):.3e}", "N")
print("Earth escape v  :", round(escape_velocity(5.972e24, 6.371e6), 2), "m/s")
print("Earth as a BH   :", round(schwarzschild_radius(5.972e24), 6), "m")

## 5. Graph algorithms

CDS graphs use **integer vertices** (a dense array model). Build one with `Graph(n_vertices=..., directed=...)`.

In [ ]:
from cds.graph import Graph, dijkstra

g = Graph(n_vertices=5, directed=False)
for u, v, w in [(0, 1, 4), (0, 2, 2), (1, 2, 1), (1, 3, 5), (2, 4, 10), (3, 4, 2)]:
    g.add_edge(u, v, weight=w)

dists, prev = dijkstra(g, 0)
print("cost 0 -> 4:", dists[4])

In [ ]:
from cds.graph import Graph, has_cycle, kruskal_mst, topological_sort

g = Graph(n_vertices=4, directed=False)
for u, v, w in [(0, 1, 1), (1, 2, 2), (0, 2, 3), (2, 3, 5)]:
    g.add_edge(u, v, weight=w)

mst_edges, total = kruskal_mst(g)
print("MST edges:", mst_edges, "total:", total)
print("has_cycle:", has_cycle(g))

# 0=compile, 1=test, 2=package, 3=deploy
dag = Graph(n_vertices=4, directed=True)
for u, v in [(0, 1), (1, 2), (2, 3)]:
    dag.add_edge(u, v, weight=1)
print("build order:", topological_sort(dag))

## 6. Signal processing

Frequency analysis and filtering, with no FFT library underneath.

In [ ]:
import math

from cds.signals import power_spectrum

n = 200
signal = [math.sin(2 * math.pi * 50 * i / 1000) + 0.3 * i / n for i in range(n)]
freqs = power_spectrum(signal)
peak = max(range(len(freqs)), key=lambda k: freqs[k])
print("dominant bin:", peak)  # a 50 Hz tone sampled at 1000 Hz

In [ ]:
from cds.signals import apply_filter, butter_lowpass, fft_radix2

x = [math.sin(2 * math.pi * 3 * k / 64) for k in range(64)]  # exactly 3 cycles
spec = fft_radix2(x)
print("energy in bin 3:", round(abs(spec[3]), 3))

coeffs = butter_lowpass(order=4, cutoff=0.2)
smoothed = apply_filter([1, 2, 5, 3, 8, 6, 9, 4], coeffs)
print("smoothed:", [round(v, 3) for v in smoothed])

## 7. Statistics and hypothesis testing

Descriptive statistics, regression, significance tests and effect sizes.

In [ ]:
from cds.stats import correlation, linear_regression, mean, stdev

a = [12.5, 14.3, 11.8, 15.1, 13.7]
b = [3.1, 4.0, 2.9, 4.4, 3.8]
print("mean/stdev :", round(mean(a), 2), round(stdev(a), 3))
print("correlation:", round(correlation(a, b), 4))

x = [1, 2, 3, 4, 5]
y = [2.1, 3.9, 6.2, 7.8, 10.1]
reg = linear_regression(x, y)
print(f"y = {reg.slope:.3f}x + {reg.intercept:.3f}  (r2={reg.r_squared:.3f})")

In [ ]:
from cds.stats import chi_square_gof, cohens_d, one_sample_ttest, two_sample_ttest

sample = [4.8, 5.1, 4.9, 5.0, 5.2]
print("one-sample p  :", round(one_sample_ttest(sample, popmean=5.0).p_value, 4))

group_a = [22, 25, 19, 24, 23]
group_b = [28, 30, 27, 31, 29]
print("two-sample p  :", round(two_sample_ttest(group_a, group_b).p_value, 6))

observed = [8, 9, 12, 7, 11, 13]
print("die fairness p:", round(chi_square_gof(observed, expected=[10] * 6).p_value, 4))
print("Cohen's d     :", round(cohens_d([1, 2, 3, 4], [4, 5, 6, 7]), 4))

## 8. Time-series analysis

Smoothing, decomposition, stationarity tests and forecasting.

In [ ]:
from cds.stats import difference, moving_average, seasonal_decompose

series = [10 + 0.5 * i + 3 * math.sin(i / 2) for i in range(48)]
print("moving average:", [round(v, 2) for v in moving_average(series, window=4)[:4]])
stationary = difference(series)
trend, seasonal, residual = seasonal_decompose(series, period=12)
print("decomposition lengths:", len(trend), len(seasonal), len(residual))

In [ ]:
from cds.stats import (
    autocorrelation_function,
    exponential_smoothing,
    kpss_statistic,
    ljung_box,
)

kpss = kpss_statistic(series)
print("KPSS stat/p:", round(kpss.statistic, 4), round(kpss.p_value, 4))
print("Ljung-Box p:", round(ljung_box(series, lags=10).p_value, 6))
print("ACF(0..2)  :", [round(a, 3) for a in autocorrelation_function(series, max_lag=12)[:3]])
print("forecast   :", [round(f, 2) for f in exponential_smoothing(series, alpha=0.3)[-3:]])

## 9. Monte Carlo

Random sampling with an explicit error bar on every estimate.

In [ ]:
from cds.montecarlo import estimate_pi, mc_integrate, random_walk_2d

pi_result = estimate_pi(n_samples=50_000)
print(f"pi = {pi_result.estimate:.5f} +/- {pi_result.std_error:.5f}")

# Integral of sin(x) over 0..pi is exactly 2.
r = mc_integrate(math.sin, 0.0, math.pi, n_samples=200_000)
print(f"int sin = {r.estimate:.4f} +/- {r.std_error:.4f}")

print("random walk points:", len(random_walk_2d(steps=1000)))  # start + 1000 steps

## 10. Numerical integration (quadrature)

Deterministic quadrature rules, including adaptive refinement for spiky integrands.

In [ ]:
from cds.numerical_integration import gaussian_quadrature, simpson, trapezoid


def f(x):
    return math.exp(-(x**2))  # integral over 0..1 ~ 0.746824


print("trapezoid:", round(trapezoid(f, 0.0, 1.0, n=100), 8))
print("simpson  :", round(simpson(f, 0.0, 1.0, n=100), 8))
print("gauss    :", round(gaussian_quadrature(f, 0.0, 1.0, n=5), 8))

In [ ]:
from cds.numerical_integration import adaptive_simpson, gaussian_quadrature_2d, simpson_2d


# A narrow Gaussian peak: adaptive refinement concentrates effort near it.
def spiky(x):
    return math.exp(-((x - 0.5) ** 2) / 0.001)


res = adaptive_simpson(spiky, 0.0, 1.0, tol=1e-8)
print("adaptive  :", round(res.value, 6), "| evaluations:", res.n_eval)


# Gaussian bump over the unit square.
def g(x, y):
    return math.exp(-(x**2 + y**2))


print("simpson_2d:", round(simpson_2d(g, 0.0, 1.0, 0.0, 1.0, nx=20, ny=20), 8))
print("gauss_2d  :", round(gaussian_quadrature_2d(g, 0.0, 1.0, 0.0, 1.0, n=4), 8))

## 11. Differential equations

Explicit, adaptive and implicit solvers side by side.

In [ ]:
from cds.diffeq import rk4, rk45


# dy/dt = -y, y(0)=1  =>  y(2) = e^-2 ~ 0.135335
def decay(t, y):
    return -y


sol = rk4(decay, t0=0.0, y0=1.0, t_end=2.0, dt=0.01)
print("rk4  y(2):", round(sol.y[-1], 6), "steps:", len(sol.t))


# Logistic growth: adaptive RK45 takes bigger steps where it can.
def logistic(t, y):
    return 0.5 * y * (1 - y / 10)


adaptive = rk45(logistic, t0=0.0, y0=0.5, t_end=20.0, rtol=1e-6)
print("rk45 y(20):", round(adaptive.y[-1], 6), "(carrying capacity 10)")

In [ ]:
from cds.diffeq import backward_euler, euler_method, solve_system


# Lotka-Volterra: dx/dt = ax - bxy, dy/dt = dxy - cy
def lotka(t, state):
    x, y = state
    return [1.1 * x - 0.4 * x * y, 0.1 * x * y - 0.4 * y]


ts, ys = solve_system(lotka, t0=0.0, y0=[10.0, 5.0], t_end=40.0, dt=0.05)
print("trajectory length:", len(ys), "final state:", [round(v, 2) for v in ys[-1]])


# Stiff decay: explicit Euler diverges, backward Euler does not.
def stiff(t, y):
    return -1000.0 * (y - 1.0)


print("backward euler:", round(backward_euler(stiff, 0.0, 0.0, 0.05, dt=0.01).y[-1], 4))
print("explicit euler:", round(euler_method(stiff, 0.0, 0.0, 0.05, dt=0.01).y[-1], 1), "(diverged)")

## 12. Optimization

Gradient descent, Adam and Newton's method on scalar and vector objectives.

In [ ]:
import math

from cds.optimization import gradient_descent, newton_method


def bowl(v):
    return (v[0] - 1) ** 2 + 100 * (v[1] - v[0] ** 2) ** 2


r = gradient_descent(lambda x: (x - 3) ** 2, x0=0.0, lr=0.1)
print("min of (x-3)^2:", round(r.x, 6), round(r.value, 8), "converged:", r.converged)

root = newton_method(math.cos, x0=1.0)
print("cos root:", round(root.x, 6), "vs pi/2 =", round(math.pi / 2, 6))

multi = gradient_descent(bowl, x0=[0.0, 0.0], lr=0.001, max_iter=50000)
print("Rosenbrock min:", [round(v, 3) for v in multi.x])

## 13. Hypothesis generation

Structured, offline hypothesis generation and evaluation against observed data — no network calls.

In [ ]:
from cds.hypothesis import generate_hypotheses

hypos = generate_hypotheses(
    "Why is the measured Hubble constant lower in some methods?",
    domain="cosmology",
    n=3,
)
for cand in hypos:
    print("-", cand.statement, "=>", cand.predictions)

In [ ]:
from cds.core.models import Domain, Hypothesis, HypothesisStatus
from cds.hypothesis import ChiSquareGofPayload, HypothesisEvaluator

die = Hypothesis(
    id="H-die",
    statement="a six-sided die is fair",
    domain=Domain.MATHEMATICS,
    research_question="is the die fair?",
    status=HypothesisStatus.TESTABLE,
)
payload: ChiSquareGofPayload = {
    "observed": [8, 12, 10, 11, 9, 10],
    "expected": [10, 10, 10, 10, 10, 10],
}
result = HypothesisEvaluator().evaluate(die, {"chi_square_gof": payload})
print(f"p = {result.p_value:.4f}  significant = {result.is_significant}")
print(result.conclusion)

## 14. Knowledge organization

A concept graph, research notes, and ranked search across both.

In [ ]:
from cds.knowledge import KnowledgeGraph, Notebook, search

kg = KnowledgeGraph(name="mechanics")
kg.add_concept("momentum", description="p = m*v")
kg.add_concept("force", description="F = dp/dt")
kg.add_relation("force", "momentum", "depends-on")
print("neighbors of force:", kg.neighbors("force"))

nb = Notebook(name="mechanics")
nb.add_note(
    note_id="n1",
    title="Newton II",
    body="F equals the rate of change of momentum.",
    linked_concepts=["force", "momentum"],
)
print("search hits:", len(search(kg, nb, query="rate of change")))

---

## Where next?

- 📖 The full recipe book: [docs/cookbook.md](https://furox-art.github.io/scientific-computing-system/cookbook/) — this notebook walks through one recipe per domain; the Markdown file has the rest.
- 🧭 Guided walkthrough: [A Tour of Numerical Methods](https://furox-art.github.io/scientific-computing-system/tour_of_numerical_methods/), also available as a notebook.
- 📚 Full public surface: [API Reference](https://furox-art.github.io/scientific-computing-system/api/).
- 🧪 The same snippets stand alone in [`examples/`](https://github.com/Furox-Art/scientific-computing-system/tree/main/examples).

> **Experiment:** change the temperature delta in §1, the tone frequency in §6, or the die counts in §13. Nothing here is locked — that is the point of pure, readable code.